# Human vs AI Trivia Challenge

**dc_dalin - Week 6 Contribution**

Compare human vs AI performance on trivia questions across multiple domains.

In [ ]:
import os
import json
import time
import random
from datetime import datetime
from typing import Dict, List, Tuple

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [ ]:
# Load questions
with open('questions_dataset.json', 'r') as f:
    data = json.load(f)
    questions = data['questions']
    metadata = data['metadata']

print(f"Loaded {len(questions)} questions | Author: {metadata['author']}")

In [ ]:
def get_ai_answer(question: str, choices: List[str], timeout: int = 5) -> Tuple[str, float, bool]:
    start_time = time.time()
    prompt = f"""Answer this trivia question with just the answer:

Question: {question}

Choices:
{chr(10).join([f'{chr(65+i)}. {choice}' for i, choice in enumerate(choices)])}

Provide only the answer."""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=50,
            timeout=timeout,
            temperature=0,
            messages=[{"role": "user", "content": prompt}]
        )
        elapsed = (time.time() - start_time) * 1000
        return response.choices[0].message.content.strip(), elapsed, True
    except Exception as e:
        return f"Error: {str(e)}", (time.time() - start_time) * 1000, False

In [ ]:
def check_answer(given: str, correct: str) -> bool:
    given, correct = given.lower().strip(), correct.lower().strip()
    if given == correct or correct in given or given in correct:
        return True
    
    # Fuzzy matching
    given_words = set(w for w in given.split() if w not in ['the', 'a', 'an', 'of', 'and'])
    correct_words = set(w for w in correct.split() if w not in ['the', 'a', 'an', 'of', 'and'])
    
    if correct_words and len(given_words.intersection(correct_words)) / len(correct_words) >= 0.7:
        return True
    return False

In [ ]:
# Game state
game_state = {
    'current_idx': 0,
    'questions': [],
    'results': [],
    'human_score': 0,
    'ai_score': 0,
    'num_questions': 10
}

def start_game(num: int, difficulty: str, category: str) -> Tuple:
    filtered = questions.copy()
    if difficulty != "All":
        filtered = [q for q in filtered if q['difficulty'] == difficulty]
    if category != "All":
        filtered = [q for q in filtered if q['category'] == category]
    
    if len(filtered) < num:
        return (
            f"Only {len(filtered)} questions available. Adjust filters.",
            gr.update(visible=False), gr.update(visible=True),
            "", "", "", "", gr.update(visible=False)
        )
    
    game_state.update({
        'current_idx': 0,
        'questions': random.sample(filtered, num),
        'results': [],
        'human_score': 0,
        'ai_score': 0,
        'num_questions': num
    })
    
    q = game_state['questions'][0]
    return (
        f"Question 1/{num}",
        gr.update(visible=True),
        gr.update(visible=False),
        q['question'],
        gr.Radio(choices=q['choices'], label="Your answer:"),
        f"**{q['category']}** | {q['difficulty']}",
        "",
        gr.update(visible=True)
    )

def submit_answer(human_answer: str) -> Tuple:
    if not human_answer:
        return "Select an answer!", "", gr.update(), gr.update()
    
    idx = game_state['current_idx']
    q = game_state['questions'][idx]
    
    human_correct = check_answer(human_answer, q['answer'])
    ai_answer, ai_time, ai_success = get_ai_answer(q['question'], q['choices'])
    ai_correct = check_answer(ai_answer, q['answer']) if ai_success else False
    
    if human_correct:
        game_state['human_score'] += 1
    if ai_correct:
        game_state['ai_score'] += 1
    
    game_state['results'].append({
        'question': q['question'],
        'category': q['category'],
        'difficulty': q['difficulty'],
        'correct_answer': q['answer'],
        'human_answer': human_answer,
        'human_correct': human_correct,
        'ai_answer': ai_answer,
        'ai_correct': ai_correct,
        'ai_time_ms': ai_time
    })
    
    feedback = f"""
**Correct**: {q['answer']}

**You**: {human_answer} {'✅' if human_correct else '❌'}  
**AI**: {ai_answer} {'✅' if ai_correct else '❌'} ({ai_time:.0f}ms)

**Score**: Human {game_state['human_score']} - {game_state['ai_score']} AI

*{q['explanation']}*
"""
    
    is_last = idx + 1 >= game_state['num_questions']
    return (
        feedback,
        "",
        gr.update(visible=False),
        gr.update(visible=True, value="View Results" if is_last else "Next ➡️")
    )

def next_question() -> Tuple:
    game_state['current_idx'] += 1
    idx = game_state['current_idx']
    
    # Check if this was the last question
    if idx >= game_state['num_questions']:
        # Show results
        h_score = game_state['human_score']
        a_score = game_state['ai_score']
        total = game_state['num_questions']
        
        winner = "🏆 Human Wins" if h_score > a_score else "🤖 AI Wins" if a_score > h_score else "🤝 Tie"
        
        df = pd.DataFrame(game_state['results'])
        avg_time = df['ai_time_ms'].mean()
        
        # Build category table manually
        cat_stats = df.groupby('category')[['human_correct', 'ai_correct']].sum()
        cat_table = "\n".join([f"- **{cat}**: Human {int(row['human_correct'])} | AI {int(row['ai_correct'])}" 
                               for cat, row in cat_stats.iterrows()])
        
        summary = f"""
# {winner}

**Human**: {h_score}/{total} ({h_score/total*100:.1f}%)  
**AI**: {a_score}/{total} ({a_score/total*100:.1f}%)

**AI Avg Response**: {avg_time:.0f}ms

### By Category
{cat_table}

*dc_dalin - Week 6*
"""
        
        fig = create_charts(df, h_score, a_score, total)
        
        # Save results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        with open(f"results_{timestamp}.json", 'w') as f:
            json.dump({'scores': {'human': h_score, 'ai': a_score}, 'results': game_state['results']}, f, indent=2)
        
        return (
            "",
            "",
            gr.update(choices=[], value=None),
            "",
            "",
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
            summary,
            fig
        )
    
    # Load next question
    q = game_state['questions'][idx]
    return (
        f"Question {idx + 1}/{game_state['num_questions']}",
        q['question'],
        gr.Radio(choices=q['choices'], label="Your answer:", value=None),
        f"**{q['category']}** | {q['difficulty']}",
        "",
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=True),
        gr.update(visible=False),
        "",
        None
    )

def create_charts(df: pd.DataFrame, h: int, a: int, total: int):
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Accuracy", "By Category", "Distribution", "AI Response Time"),
        specs=[[{"type": "bar"}, {"type": "bar"}], [{"type": "pie"}, {"type": "scatter"}]]
    )
    
    fig.add_trace(go.Bar(x=["Human", "AI"], y=[h/total*100, a/total*100], 
                         marker_color=["#3498db", "#e74c3c"]), row=1, col=1)
    
    cat_stats = df.groupby('category')[['human_correct', 'ai_correct']].sum()
    fig.add_trace(go.Bar(name="Human", x=cat_stats.index, y=cat_stats['human_correct'], 
                         marker_color="#3498db"), row=1, col=2)
    fig.add_trace(go.Bar(name="AI", x=cat_stats.index, y=cat_stats['ai_correct'], 
                         marker_color="#e74c3c"), row=1, col=2)
    
    both_correct = sum(df['human_correct'] & df['ai_correct'])
    human_only = h - both_correct
    ai_only = a - both_correct
    both_wrong = total - h - a + both_correct
    
    fig.add_trace(go.Pie(labels=["Human Only", "AI Only", "Both Wrong"], 
                         values=[human_only, ai_only, both_wrong],
                         marker_colors=["#3498db", "#e74c3c", "#95a5a6"]), row=2, col=1)
    
    fig.add_trace(go.Scatter(x=list(range(1, len(df)+1)), y=df['ai_time_ms'], 
                            mode="lines+markers", marker=dict(size=8, color="#e74c3c")), row=2, col=2)
    
    fig.update_layout(height=800, showlegend=False, title_text="Performance Analysis (dc_dalin)")
    return fig

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="Human vs AI Trivia") as demo:
    gr.Markdown("# 🎯 Human vs AI Trivia\n\n*dc_dalin - Week 6*")
    
    with gr.Row(visible=True) as setup:
        with gr.Column():
            num = gr.Slider(5, 25, value=10, step=5, label="Questions")
            diff = gr.Dropdown(["All", "Easy", "Medium", "Hard"], value="All", label="Difficulty")
            cat = gr.Dropdown(["All"] + metadata['categories'], value="All", label="Category")
            start_btn = gr.Button("Start", variant="primary")
    
    status = gr.Markdown("")
    
    with gr.Row(visible=False) as game:
        with gr.Column():
            q_text = gr.Markdown("")
            q_meta = gr.Markdown("")
            choices = gr.Radio(label="Your answer:", choices=[])
            submit_btn = gr.Button("Submit", variant="primary", visible=True)
            next_btn = gr.Button("Next ➡️", variant="secondary", visible=False)
            feedback = gr.Markdown("")
    
    with gr.Row(visible=False) as results:
        with gr.Column():
            results_text = gr.Markdown("")
            chart = gr.Plot()
            restart = gr.Button("Play Again", variant="primary")
    
    # Event handlers
    start_btn.click(
        start_game, 
        [num, diff, cat], 
        [status, game, setup, q_text, choices, q_meta, feedback, submit_btn]
    )
    
    submit_btn.click(
        submit_answer, 
        [choices], 
        [feedback, status, submit_btn, next_btn]
    )
    
    next_btn.click(
        next_question, 
        outputs=[status, q_text, choices, q_meta, feedback, submit_btn, next_btn, game, results, results_text, chart]
    )
    
    restart.click(
        lambda: (gr.update(visible=True), gr.update(visible=False)), 
        outputs=[setup, results]
    )

demo.launch(share=False)